In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent / "src"))

from utils import load_dataset, ensure_sorted, train_test_split_time_series
from feature_engineering import add_lag_features, add_rolling_features, add_time_features
from evaluation import evaluate_forecast, print_evaluation
from models.sarimax_model import run_sarimax


2 — Load & prepare data (same as before)

In [ ]:
# Load merged dataset
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
MERGED_PATH = PROCESSED_DIR / "demand_temperature_half_hourly.csv"

df = load_dataset(MERGED_PATH)
df = ensure_sorted(df)

# Feature engineering
df_fe = df.copy()

df_fe = add_lag_features(df_fe, column='demand', lags=[1, 48, 96])
df_fe = add_rolling_features(df_fe, column='demand', windows=[48, 96, 336])
df_fe = add_time_features(df_fe)

# Drop rows created by lag/rolling
df_fe = df_fe.dropna().reset_index(drop=True)

# Train/test split
train, test = train_test_split_time_series(df_fe, test_size=0.1)

# Prepare SARIMAX inputs
train_y = train['demand']
train_exog = train[['tmean']]
test_y = test['demand']
test_exog = test[['tmean']]

train.head(), test.head()


3 — Fit SARIMAX on baseline

In [ ]:
train_y = train['demand']
train_exog = train[['tmean']]
test_exog = test[['tmean']]

sarimax_model = run_sarimax(train_y, train_exog,
                            order=(2,1,2), seasonal_order=(1,1,1,48))

baseline_forecast = sarimax_model.forecast(steps=len(test), exog=test_exog)


4 — Create scenarios

In [ ]:
scenario_warmer = test_exog.copy()
scenario_colder = test_exog.copy()

scenario_warmer['tmean'] = scenario_warmer['tmean'] + 2.0
scenario_colder['tmean'] = scenario_colder['tmean'] - 2.0

forecast_warmer = sarimax_model.forecast(steps=len(test), exog=scenario_warmer)
forecast_colder = sarimax_model.forecast(steps=len(test), exog=scenario_colder)


5 — Plot scenarios

In [ ]:
plt.figure(figsize=(15,6))
plt.plot(test['timestamp'], test['demand'], label='Actual', color='black')
plt.plot(test['timestamp'], baseline_forecast, label='Baseline', alpha=0.8)
plt.plot(test['timestamp'], forecast_warmer, label='+2°C scenario', alpha=0.8)
plt.plot(test['timestamp'], forecast_colder, label='-2°C scenario', alpha=0.8)

plt.legend()
plt.title("Scenario Analysis: Impact of Temperature on Demand")
plt.tight_layout()
plt.show()


6 — Aggregate impact

In [ ]:
scenario_df = pd.DataFrame({
    'timestamp': test['timestamp'],
    'actual': test['demand'],
    'baseline': baseline_forecast,
    '+2C': forecast_warmer,
    '-2C': forecast_colder
})

scenario_df[['baseline', '+2C', '-2C']].sum()
